In [1]:
import sys
sys.path.append("../src")
import torch
import torch.nn as nn
from collections import defaultdict
from sklearn.metrics import f1_score
from data import generate_examples, generate_examples_with_error_type
from model import DyckTransformer

VOCAB = ["(", ")", "[", "]", "[PAD]", "[CLS]", "[SEP]"]
stoi = {tok: i for i, tok in enumerate(VOCAB)}
edit_stoi = {"OK": 0, "DELETE": 1, "INSERT())": 2, "INSERT(])": 3, "REPLACE())": 4, "REPLACE(])": 5}
edit_itos = {i: l for l, i in edit_stoi.items()}
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
model = DyckTransformer(
    vocab_size=len(VOCAB),
    pad_idx=stoi["[PAD]"],
    num_edit_labels=6,
    d_model=128,
    n_heads=4,
    n_layers=2,
    max_len=80,
    dropout=0.1
)
model.load_state_dict(torch.load("../results/models/dyck_transformer.pt", map_location=device))
model.to(device)
model.eval()

DyckTransformer(
  (embed): Embedding(7, 128)
  (pos): SinusoidalPositionalEncoding()
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
        )
        (linear1): Linear(in_features=128, out_features=512, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=512, out_features=128, bias=True)
        (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (dropout): Dropout(p=0.1, inplace=False)
  (cls_head): Linear(in_features=128, out_features=2, bias=True)
  (token_head): Linear(in_features=128, out_features=6, bias=True)
)

In [4]:
cor_test_data = generate_examples(5000, task="correction")
batch_size = 64

In [5]:
total_token_correct = 0
total_token_count = 0
exact_match = 0
total_sequences = 0

with torch.no_grad():
    for i in range(0, len(cor_test_data), batch_size):
        batch = cor_test_data[i:i+batch_size]
        xs, ys = zip(*batch)
        xs = torch.tensor([[stoi[tok] for tok in x] for x in xs]).to(device)
        ys = torch.tensor([[edit_stoi[l] for l in labels] for labels in ys]).to(device)

        _, token_logits = model(xs)
        preds = torch.argmax(token_logits, dim=2)

        # token-level accuracy
        total_token_correct += (preds == ys).sum().item()
        total_token_count += ys.numel()

        # exact-match accuracy
        exact_match += (preds == ys).all(dim=1).sum().item()
        total_sequences += len(batch)

token_acc = total_token_correct / total_token_count
exact_match_acc = exact_match / total_sequences

print(f"token-level accuracy: {token_acc:.4f}")
print(f"exact-match accuracy: {exact_match_acc:.4f}")

c:\Users\sarav\projects\dyck_transformer\.pixi\envs\default\Lib\site-packages\torch\nn\modules\transformer.py:531: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. We recommend specifying layout=torch.jagged when constructing a nested tensor, as this layout receives active development, has better operator coverage, and works with torch.compile. (Triggered internally at C:\bld\libtorch_1772176602146\work\aten\src\ATen\NestedTensorImpl.cpp:182.)
  output = torch._nested_tensor_from_mask(


token-level accuracy: 0.9953
exact-match accuracy: 0.6396


In [7]:
cor_test_data_typed = generate_examples_with_error_type(5000, task="correction")

results = defaultdict(lambda: {"token_correct": 0, "token_total": 0, "exact": 0, "total": 0})

with torch.no_grad():
    for i in range(0, len(cor_test_data_typed), batch_size):
        batch = cor_test_data_typed[i:i+batch_size]
        xs = [x for x, _, _ in batch]
        ys = [y for _, y, _ in batch]
        types = [t for _, _, t in batch]
        xs = torch.tensor([[stoi[tok] for tok in x] for x in xs]).to(device)
        ys_tensor = torch.tensor([[edit_stoi[l] for l in labels] for labels in ys]).to(device)

        _, token_logits = model(xs)
        preds = torch.argmax(token_logits, dim=2)

        for j, etype in enumerate(types):
            results[etype]["token_correct"] += (preds[j] == ys_tensor[j]).sum().item()
            results[etype]["token_total"] += ys_tensor[j].numel()
            results[etype]["exact"] += (preds[j] == ys_tensor[j]).all().item()
            results[etype]["total"] += 1

print(f"{'':>6} {'token acc':>12} {'exact match':>14} {'n':>6}")
for etype in ["none", "E1", "E2", "E3", "E4"]:
    r = results[etype]
    token_acc = r["token_correct"] / r["token_total"]
    exact_acc = r["exact"] / r["total"]
    print(f"{etype:>6} {token_acc:>12.4f} {exact_acc:>14.4f} {r['total']:>6}")

          token acc    exact match      n
  none       1.0000         1.0000   2531
    E1       0.9875         0.0433    647
    E2       0.9884         0.0984    620
    E3       0.9955         0.6429    588
    E4       0.9915         0.3648    614
